# Giving an agent new tools: the Materials Project and your own Python

This is the second notebook. It starts where `01_agents_vscode_and_docker_intro.ipynb` ended: you
have a **dev container** with Claude Code and Gemini CLI, and you stay signed in after a rebuild
(section 2.6 of notebook 01). If you haven't done that yet, go back and finish it first.

As in notebook 01, every new word is explained the first time it appears, and there is a glossary
at the end.

**What you will do**

| Stage | What happens | Why |
|---|---|---|
| **1. Get a key** | Get a free Materials Project key and pass it into the container. | The database needs to know who is asking. |
| **2. Connect the database** | Install the Materials Project "plug" (an MCP server) and connect it to the agent. | The agent can now look things up in the real database instead of guessing. |
| **3. Set the rules** | Write down what the agent should do, and block what it must never do. | Useful help, without surprises. |
| **4. Write your own tool** | Turn two short Python functions into tools the agent can use. | Anything with a Python library can become a tool. |
| **5. Check and freeze** | Test everything together, then make the setup repeatable. | So it still works next month, and on a colleague's computer. |

**How to read this notebook** (same as notebook 01)

- 🖥️ **Terminal** boxes: commands you type into a terminal. Most run **inside the container**; the
  few that run on your own computer say so.
- 📄 **File** boxes: a file you create, with its name above the box.
- ▶️ **Code cells**: small checks. Run them with the container's Python to confirm each step worked.

---
# Stage 1 — Get a Materials Project key

## 1.1 Where we left off

At the end of notebook 01 you asked the agent:

```
What is the energy above hull of La3Ni2O7 in the Materials Project database,
and what is its Materials Project ID? Only answer from a source you can
actually query. If you cannot verify it, say so.
```

and it could not really answer. It either said it couldn't check, answered from memory, or quoted
a web page. The agent has no connection to the database. This notebook builds that connection.

## 1.2 What is an API, and what is an API key?

The Materials Project website is made for people. Programs use a different entrance, the **API**
(*application programming interface*). An API is a list of requests a program can send, such as
*"give me all phases in the La–Ni–O system"*, and it answers with data instead of a web page.

The **API key** is your personal password for that entrance. It is free, but it is **secret**:
never put it in a notebook, a slide or a file you share or upload to GitHub.

**Get your key:**

1. Create an account at [materialsproject.org](https://materialsproject.org).
2. Open your dashboard and copy the **API key**.

## 1.3 Save the key on your own computer

We keep the key in an **environment variable**: a named setting that every program started from
your terminal can read. This keeps the key out of your files.

This step happens on your **own computer** (the **host**), *not* inside the container. The container
will copy the value in 1.4.

🖥️ Terminal — on your own computer, macOS / Linux / Windows with WSL

Add this line at the end of the file `~/.bashrc` (or `~/.zshrc` on a Mac), with your real key:

```bash
export MP_API_KEY=your_key_here
```

Then open a **new** terminal and check:

```bash
echo $MP_API_KEY
```

🖥️ Terminal — on your own computer, Windows without WSL (PowerShell)

```powershell
setx MP_API_KEY "your_key_here"
```

Then close **every** terminal and VS Code window, and reopen them.

> ⚠️ **The most common problem in this whole notebook.** VS Code only knows the environment
> variables that existed *when VS Code was started*. If VS Code was already open, or you started it
> from the Dock or Start menu on a Mac, it may not know about `MP_API_KEY`, even though your
> terminal does.
>
> **Fix:** quit VS Code completely (on a Mac, `Cmd+Q`, not just closing the window), open a
> terminal, go to your project folder and start VS Code from there with `code .`
>
> If you skip this, nothing breaks right away. The container builds, and much later database
> requests fail with "authentication error". That makes it hard to trace back to this step.

## 1.4 Pass the key into the container

The container can't see your computer's environment variables unless you pass them in. Open
`.devcontainer/devcontainer.json` from notebook 01 and add **one line** to `containerEnv`:

📄 File: `.devcontainer/devcontainer.json`
```json
{
  "name": "Agents demo",
  "image": "mcr.microsoft.com/devcontainers/python:3.12",

  "features": {
    "ghcr.io/devcontainers/features/node:1": {},
    "ghcr.io/anthropics/devcontainer-features/claude-code:1": {}
  },

  "containerEnv": {
    "CLAUDE_CONFIG_DIR": "/home/vscode/.claude",
    "MP_API_KEY": "${localEnv:MP_API_KEY}"
  },

  "mounts": [
    "source=claude-login-${devcontainerId},target=/home/vscode/.claude,type=volume",
    "source=gemini-login-${devcontainerId},target=/home/vscode/.gemini,type=volume"
  ],

  "postCreateCommand": "sudo chown -R vscode:vscode /home/vscode/.claude /home/vscode/.gemini && npm install -g @google/gemini-cli",

  "customizations": {
    "vscode": {
      "extensions": ["ms-python.python", "ms-toolsai.jupyter"]
    }
  }
}
```

`${localEnv:MP_API_KEY}` means *"copy the value of `MP_API_KEY` from my own computer"*. The key
itself never appears in the file, so this file is still safe to share.

Now **rebuild**: Command Palette (`Ctrl+Shift+P` / `Cmd+Shift+P`) → **Dev Containers: Rebuild
Container**. Your logins survive the rebuild thanks to the volumes from notebook 01.

Then run the check below.

In [ ]:
import os, shutil
from pathlib import Path

print("inside the container?", "yes" if Path("/.dockerenv").exists() or os.environ.get("REMOTE_CONTAINERS") else "NO - reopen in container first")
print("this folder          :", Path.cwd())
key = os.environ.get("MP_API_KEY", "")
print("MP_API_KEY           :", f"set ({len(key)} characters)" if key else "MISSING -> see the warning in 1.3, then rebuild")
for program in ("claude", "gemini", "git"):
    print(f"{program:21}:", shutil.which(program) or "NOT FOUND")

---
# Stage 2 — Connect the Materials Project to the agent

## 2.1 MCP: a standard plug for tools

In notebook 01 the agent had a few built-in **tools**: read a file, write a file, run a command.
To query the Materials Project it needs a new one.

**MCP** (*Model Context Protocol*) is the standard way to add tools to an agent. Think of it as a
**USB plug for tools**:

- An **MCP server** is a small program that offers some tools. The Materials Project's server
  offers two: `search` (find materials) and `fetch` (get the details of one material).
- The agent is the **client**: it plugs into the server and can use its tools.
- Because the plug is standard, the same server works in Claude Code, Gemini CLI and other agents.

You never start the server yourself during normal use. **The agent starts it in the background**
when it launches, and they talk to each other by passing text back and forth.

```
 ┌──────────────┐   "search La-Ni-O"   ┌──────────────────┐   web request   ┌────────────────────┐
 │    agent     │ ───────────────────▶ │  MCP server      │ ──────────────▶ │ Materials Project  │
 │ (Claude Code)│ ◀─────────────────── │  (runs in the    │ ◀────────────── │ database (online)  │
 └──────────────┘    list of phases    │   container)     │   uses your key └────────────────────┘
                                       └──────────────────┘
```

## 2.2 Install the server

The server is a Python program. We install it with a short script, so the rebuild can repeat it
automatically later. Create two files in the `.devcontainer` folder.

📄 File: `.devcontainer/setup-mp-mcp.sh`
```bash
#!/usr/bin/env bash
set -euo pipefail
HERE="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"

# 1. Download the server's code (once).
SRC="$HOME/mp-mcp"
[ -d "$SRC" ] || git clone https://github.com/esoteric-ephemera/mp_api.git "$SRC"
cd "$SRC"

# 2. Use one exact, tested version of that code.
git checkout 553f725af71d48197424807b02ae8c80bc32d640

# 3. Make a private Python installation just for this server, and install it there.
python3 -m venv "$SRC/.venv"
"$SRC/.venv/bin/pip" install --upgrade pip
"$SRC/.venv/bin/pip" install -e '.[mcp]' -c "$HERE/mp-mcp-constraints.txt"

echo "MCP server interpreter: $SRC/.venv/bin/python"
```

📄 File: `.devcontainer/mp-mcp-constraints.txt`
```
emmet-core==0.86.4
```

What the script does, in plain words:

| Step | Plain meaning |
|---|---|
| `set -euo pipefail` | Stop at the first error instead of carrying on. |
| `git clone …` | Download the server's code from GitHub into `~/mp-mcp`. |
| `git checkout 553f725…` | Switch to one exact version of the code, identified by that long code (a **commit hash**). |
| `python3 -m venv …` | Create a **virtual environment**: a separate Python installation with its own packages, so the server's packages never clash with anything else. |
| `pip install -e '.[mcp]' -c …` | Install the server (with its MCP extras) into that environment, following the version limits in the constraints file. |

Run it once by hand, inside the container:

🖥️ Terminal
```bash
bash .devcontainer/setup-mp-mcp.sh
```

It takes a few minutes. The last line should print the interpreter path.

### Why the two exact versions?

Both were found the hard way. Without them, the install fails.

- **The commit hash.** The Materials Project documentation says to use a version called `v0.46.0`,
  but that name doesn't exist in this copy of the code. The main version of the code doesn't include
  the MCP server at all. So we name the exact version that works.
- **`emmet-core==0.86.4`.** The server needs a package called `emmet-core` and accepts *any* newer
  version. Version 0.87.0 moved something the server relies on, and the server then crashes on
  start. The constraints file keeps it at 0.86.4.

Fixing versions like this is called **pinning**. You will do it for *everything* in Stage 5.

### Make the rebuild do it automatically

Add the script to the end of `postCreateCommand` in `devcontainer.json`, so every rebuild installs
the server by itself:

```json
"postCreateCommand": "sudo chown -R vscode:vscode /home/vscode/.claude /home/vscode/.gemini && npm install -g @google/gemini-cli && bash .devcontainer/setup-mp-mcp.sh",
```

Now check that the server's Python can load the server:

In [ ]:
# Can the server's own Python load the server?
import os, subprocess
from pathlib import Path

PY = Path.home() / "mp-mcp/.venv/bin/python"
if not PY.exists():
    print("Not installed yet -> run: bash .devcontainer/setup-mp-mcp.sh")
else:
    # Jupyter sets MPLBACKEND to its own plotting backend, which the server's Python doesn't have.
    # Remove it, so the server starts the same way it does when the agent launches it.
    env = {k: v for k, v in os.environ.items() if k != "MPLBACKEND"}
    r = subprocess.run([str(PY), "-c", "import mp_api.mcp.server; print('server loads OK')"],
                       capture_output=True, text=True, env=env)
    print(r.stdout or r.stderr[-1500:])
    if "BSPathType" in r.stderr:
        print("-> the emmet-core pin did not work. Fix with:")
        print(f"   {PY.parent}/pip install 'emmet-core==0.86.4'")

## 2.3 Tell the agent about the server

The agent needs to know that the server exists and how to start it. Claude Code looks for a file
called `.mcp.json` in the **project folder** (next to this notebook, *not* inside `.devcontainer`):

📄 File: `.mcp.json`
```json
{
  "mcpServers": {
    "materials-project": {
      "command": "/home/vscode/mp-mcp/.venv/bin/python",
      "args": ["-m", "mp_api.mcp.server"],
      "env": {
        "MP_API_KEY": "${MP_API_KEY}"
      }
    }
  }
}
```

| Entry | Plain meaning |
|---|---|
| `"materials-project"` | The name you will see in the agent. Any name works. |
| `command` | Which program to start: the Python **inside the server's virtual environment**. Write the full path. A plain `python` would pick a different Python that doesn't have the server installed, and the server would quietly fail. |
| `args` | What to give that Python: *"run the module `mp_api.mcp.server`"*. |
| `env` | Settings passed to the server. `${MP_API_KEY}` is filled in by Claude Code from the container's environment, so the real key never appears in this file. |

Check that the file is valid (a missing comma or quote makes Claude Code ignore it silently):

In [ ]:
import json
from pathlib import Path

cfg = Path.cwd() / ".mcp.json"
if not cfg.exists():
    print("No .mcp.json here - create it in the project folder (next to this notebook).")
else:
    try:
        data = json.loads(cfg.read_text())
    except json.JSONDecodeError as exc:
        raise SystemExit(f"INVALID JSON: {exc}  <- fix this first, the agent will ignore the file")
    for name, srv in data["mcpServers"].items():
        exe = Path(srv["command"])
        print(name)
        print("  command:", exe, "->", "exists" if exe.exists() else "MISSING")
        print("  args   :", srv.get("args"))
        print("  env    :", list(srv.get("env", {})))

## 2.4 Test the server by hand (once)

Start the server yourself, exactly as the agent would:

🖥️ Terminal
```bash
$HOME/mp-mcp/.venv/bin/python -m mp_api.mcp.server
```

You should see a banner and then **nothing**. That is correct: the server is waiting for an agent
to send it text. It is not frozen.

To stop it, press **Ctrl-D**. (`Ctrl-C` often doesn't work here, because the server is busy
waiting for input.)

## 2.5 Use it

1. Start Claude Code again (`claude`). Servers are connected only when the agent starts.
2. Claude asks whether you trust the servers listed in `.mcp.json`. Choose **yes**.
3. Type `/mcp`. You should see **materials-project ✔ connected** with two tools, `search` and
   `fetch`.

Now **ask the La3Ni2O7 question from 1.1 again.** This time you should see a tool call to `search`
or `fetch`, followed by a Materials Project ID (`mp-…`) and a number that came from the database.

Compare with the answer in notebook 01. The model is the same. What changed is that it has a
tool connected to the database.

> **Gemini CLI too (optional).** Gemini reads the same kind of list from `.gemini/settings.json` in
> the project folder. Put the same `"mcpServers": { … }` block in that file, restart `gemini`, and
> type `/mcp`. The server doesn't care which agent is using it.

---
# Stage 3 — Set the rules

The agent now reaches further than before. This stage uses two files that do two different jobs:

| File | Job | Strength |
|---|---|---|
| `CLAUDE.md` | Tells the agent **what it should do**: your conventions and working style. | A *request*. The agent follows it because it cooperates, but it can forget or misread it. |
| `.claude/settings.json` | Defines **what it can't do**: commands and files that are blocked. | A *hard rule*. Claude Code checks it before every action, and no prompt can get around it. |

Use both: the first so the agent doesn't try, the second so it can't.

## 3.1 `CLAUDE.md`: your instructions

Claude Code reads this file automatically at the start of every session, so you stop repeating
yourself. Create it in the project folder:

📄 File: `CLAUDE.md`
```markdown
# CLAUDE.md

Teaching environment for a seminar on AI agents in computational materials
science. Small, fast calculations only.

- Do not modify `.devcontainer/`, `.mcp.json`, or `.claude/` without asking.

## Project conventions

### Structures
- Structures go in `./structures`, as POSCAR files.

### DFT settings
- VASP 6.x, PBEsol, PAW_PBE 54 pseudopotentials.
- Default ENCUT 520 eV; default k-spacing 0.03 2π/Å.
- Never change the functional or pseudopotential set without asking.
- Convergence is not assumed — state what was checked.

### Job submission
- Never submit anything to a queue. Never run `sbatch` or `srun`.
- When a calculation needs submitting, show me the script instead.

### Working style
- Propose a plan and wait for approval before editing files.
- Make one change at a time.
- Do not invent numbers, outputs, database records or commands.
- Say clearly whether a value comes from the database, the literature,
  a calculation, or an estimate.

### Materials Project
- Use the `materials-project` tools for database questions.
- Always give the Materials Project ID for database values.
- Never invent Materials Project IDs or properties.
```

Replace the DFT settings with your group's real conventions. Gemini CLI reads a file called
`GEMINI.md` for the same purpose. You can copy the same text into it.

## 3.2 `.claude/settings.json`: hard limits

Create a folder `.claude` in the project folder, and this file inside it:

📄 File: `.claude/settings.json`
```json
{
  "$schema": "https://json.schemastore.org/claude-code-settings.json",
  "permissions": {
    "disableBypassPermissionsMode": "disable",
    "blockReadsOutsideWorkingDirectories": true,
    "deny": [
      "Read(./.env)",
      "Read(~/.ssh/**)",
      "Bash(sbatch:*)",
      "Bash(srun:*)",
      "Bash(scancel:*)"
    ]
  }
}
```

| Entry | Plain meaning |
|---|---|
| `disableBypassPermissionsMode` | Claude Code has a mode that skips all permission questions. This turns it off for the project. (The value is the word `"disable"`, not `true`.) |
| `blockReadsOutsideWorkingDirectories` | Blocks reading files outside the project folder. |
| `deny` | A list of things that are always refused. Here: reading secret files (`.env`, SSH keys) and the cluster commands that submit or cancel jobs (`sbatch`, `srun`, `scancel`). |

There is also a personal version, `.claude/settings.local.json`. It collects the permissions *you*
approve with "always allow" and isn't meant to be shared.

Check both files:

In [ ]:
import json
from pathlib import Path

for p in (Path("CLAUDE.md"), Path(".claude/settings.json")):
    if not p.exists():
        print(f"{p}: MISSING")
        continue
    print(f"{p}: {p.stat().st_size} bytes")
    if p.suffix == ".json":
        try:
            cfg = json.loads(p.read_text())
            print("   valid JSON; blocked:", cfg.get("permissions", {}).get("deny", []))
        except json.JSONDecodeError as exc:
            print("   INVALID JSON:", exc)

## 3.3 Check what the agent can actually reach

Start `claude`, type `/mcp`, and read the list carefully.

If your Claude account has **connectors** switched on at claude.ai (Gmail, Google Drive, Google
Calendar…), they appear here too, even though nothing in the container asked for them. Claude Code
brings them in from your account.

That matters before you show a session on a projector: the agent could read your email. To turn
them off:

- **In this project:** switch each one off in the `/mcp` screen. This is remembered, even after a
  rebuild.
- **For everyone who uses this project:** add `"disableClaudeAiConnectors": true` to
  `.claude/settings.json`.
- **For one session only:** start the agent with `ENABLE_CLAUDEAI_MCP_SERVERS=false claude`

It's easy to be wrong about what your agent can reach, even when nothing has gone wrong. Check
before you rely on it.

---
# Stage 4 — Write your own tool

The Materials Project server gives *information* about materials, but not the atomic positions.
To go from *"La3Ni2O7 is in the database"* to *"here are my VASP input files"*, we write two tools of
our own.

**A tool is just a Python function** with two extras:

- A **docstring**, the text in triple quotes at the start of the function. The agent reads it to
  understand what the tool does and when to use it, so write it for a reader.
- The label **`@mcp.tool`** on the line above the function (a **decorator**). It means *"offer this
  function to the agent"*. Without it, the function exists but the agent never sees it.

Create a folder `tools` in the project folder, and this file inside it:

📄 File: `tools/vasp_inputs_server.py`
```python
import os
from pathlib import Path

from fastmcp import FastMCP
from mp_api.client import MPRester
from pymatgen.core import Structure
from pymatgen.io.vasp.sets import MPRelaxSet

mcp = FastMCP("VASP Input Builder")


@mcp.tool
def get_mp_structure(mp_id: str, out_dir: str = "structures") -> dict:
    """Download a crystal structure from the Materials Project as a POSCAR.

    Use this when you need atomic coordinates — the materials-project
    server only returns summary information, not structures.

    Args:
        mp_id: Materials Project ID, e.g. "mp-18926"
        out_dir: directory to write the POSCAR into
    """
    with MPRester(os.environ["MP_API_KEY"]) as mpr:
        struct = mpr.get_structure_by_material_id(mp_id)
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    path = out / f"{mp_id}_POSCAR"
    struct.to(filename=str(path), fmt="poscar")
    return {"path": str(path),
            "formula": struct.composition.reduced_formula,
            "spacegroup": struct.get_space_group_info()[0],
            "n_sites": len(struct),
            "lattice_abc": [round(x, 4) for x in struct.lattice.abc]}


@mcp.tool
def write_mp_relax_inputs(poscar_path: str, out_dir: str,
                          encut: float | None = None) -> dict:
    """Write Materials-Project-standard VASP relaxation inputs for a structure.

    Uses pymatgen's MPRelaxSet — the same settings the Materials Project
    uses. Writes INCAR, KPOINTS, POSCAR and POTCAR.spec. Never writes
    POTCAR data, and never submits anything to a queue.

    Args:
        poscar_path: path to a POSCAR or CIF file
        out_dir: directory to write the input files into
        encut: optional plane-wave cutoff in eV
    """
    struct = Structure.from_file(poscar_path)
    vis = MPRelaxSet(struct, user_incar_settings={"ENCUT": encut} if encut else {})
    out = Path(out_dir)
    vis.write_input(str(out), potcar_spec=True)
    return {"dir": str(out),
            "files": sorted(p.name for p in out.iterdir()),
            "INCAR": (out / "INCAR").read_text()}


if __name__ == "__main__":
    mcp.run()
```

What is new here:

| Part | Plain meaning |
|---|---|
| `FastMCP("VASP Input Builder")` | Creates an MCP server with that name. `fastmcp` is a small library that does all the MCP work for you. |
| `get_mp_structure` | Downloads a structure with `MPRester` (the Materials Project's Python library) and saves it as a POSCAR. |
| `write_mp_relax_inputs` | Uses `pymatgen` to write a standard VASP relaxation input set. |
| `mcp.run()` | Starts the server when the agent launches this file. |

> **About POTCARs.** VASP's pseudopotential files (POTCARs) are licensed and must not be shared.
> `potcar_spec=True` writes only a `POTCAR.spec` file, which is the *list* of pseudopotentials to use,
> with none of the licensed data. Never put POTCARs in a container or a repository.

Now add the new server to `.mcp.json`, next to the first one. It uses the same Python, which
already has `fastmcp`, `mp_api` and `pymatgen` installed:

📄 File: `.mcp.json`
```json
{
  "mcpServers": {
    "materials-project": {
      "command": "/home/vscode/mp-mcp/.venv/bin/python",
      "args": ["-m", "mp_api.mcp.server"],
      "env": { "MP_API_KEY": "${MP_API_KEY}" }
    },
    "vasp-inputs": {
      "command": "/home/vscode/mp-mcp/.venv/bin/python",
      "args": ["/workspaces/agents-demo/tools/vasp_inputs_server.py"],
      "env": { "MP_API_KEY": "${MP_API_KEY}" }
    }
  }
}
```

If your project folder isn't called `agents-demo`, change that path (run `pwd` in the terminal
to see it). Watch the comma between the two servers: without it, the whole file is ignored.

Check that every function in your file carries the `@mcp.tool` label:

In [ ]:
# Every function should say "tool". A function without @mcp.tool is invisible to the agent.
import ast
from pathlib import Path

src = Path("tools/vasp_inputs_server.py")
if not src.exists():
    print("tools/vasp_inputs_server.py not found")
else:
    for node in ast.parse(src.read_text()).body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            is_tool = any("mcp.tool" in ast.unparse(d) for d in node.decorator_list)
            print(("tool        " if is_tool else "NOT A TOOL  ") + node.name + "()")

---
# Stage 5 — Check everything, then freeze it

## 5.1 Test the whole chain

Restart `claude` and type `/mcp`. You should now see **two** servers:

- `materials-project`, with `search` and `fetch`
- `vasp-inputs`, with `get_mp_structure` and `write_mp_relax_inputs`

**Test 1: does the key reach the database?**

```
What phases exist in the La-Ni-O system? Give me formula, material ID,
space group and energy above hull, sorted by energy above hull.
```

A table of phases means it works. If the key is missing, the error appears *here*, not when the
agent starts. That's why this test matters.

**Test 2: can the agent combine tools?**

```
Fetch La3Ni2O7 from Materials Project, save the structure to ./structures,
and build an MP-standard VASP relaxation input set for it in ./relax.
Show me the INCAR. Don't submit anything.
```

Watch the tool calls: search the database, download the structure, write the inputs. Nobody
wrote a *"Materials Project to VASP"* function. The agent chained two tools it was never told to
combine.

Now look at what it produced:

In [ ]:
from pathlib import Path

for folder in ("structures", "relax"):
    p = Path(folder)
    if not p.exists():
        print(f"{folder}/  (nothing yet)")
        continue
    print(f"{folder}/")
    for f in sorted(p.rglob("*")):
        if f.is_file():
            print(f"  {f.stat().st_size:>8} bytes  {f.relative_to(p)}")

In [ ]:
# Read the INCAR yourself. Don't rely on the agent's summary of it.
from pathlib import Path

incars = sorted(Path("relax").rglob("INCAR")) if Path("relax").exists() else []
if not incars:
    print("no INCAR yet")
else:
    text = incars[0].read_text()
    print(incars[0], "\n")
    print(text)
    tags = [line.split("=")[0].strip() for line in text.splitlines() if "=" in line]
    repeated = {t for t in tags if tags.count(t) > 1}
    print("\ntags that appear twice:", repeated or "none")

## 5.2 Read the INCAR like a referee

Every file is correct in the sense that nothing was made up. That doesn't make it right for *your*
material.

For example, look at `ISMEAR` and `ISIF`. The Materials Project settings often use `ISMEAR = -5`
(tetrahedron method) together with `ISIF = 3` (relax the cell shape and volume). For an
**insulator** that is a sensible default. For a **metal**, that combination is the one people warn
about: the tetrahedron method gives good energies but less reliable forces and stresses, and
`ISIF = 3` relies on the stress. pymatgen has a separate `MPMetalRelaxSet` for metals.

So: is your La3Ni2O7 a metal or an insulator? The agent won't ask. The tag and the tool are the
same in both cases. The answer is right for one material and wrong for the other, and only your
physics knowledge tells which. Invented facts are easy to spot. This kind of mistake is harder to
spot and more common.

## 5.3 Freeze the versions

Right now everything works because a particular set of package versions happens to fit together.
Save that set while it works. Run this in the container terminal:

🖥️ Terminal
```bash
$HOME/mp-mcp/.venv/bin/pip freeze --exclude-editable \
  | grep -v '^mp-api' > .devcontainer/mp-mcp-constraints.txt
```

`pip freeze` lists every installed package with its exact version. The two filters remove the
server itself, which can't appear in a constraints file. The setup script from 2.2 already reads
this file, so every future rebuild installs exactly these versions.

Check the result:

In [ ]:
from pathlib import Path

c = Path(".devcontainer/mp-mcp-constraints.txt")
if not c.exists():
    print("constraints file not created yet")
else:
    lines = [ln for ln in c.read_text().splitlines() if ln.strip()]
    print(f"{len(lines)} packages pinned")
    bad = [ln for ln in lines if ln.startswith("-e") or "@" in ln or "mp-api" in ln]
    print("lines that will break the install:", bad or "none")
    print("emmet:", [ln for ln in lines if ln.lower().startswith("emmet")])

## 5.4 Prove it is repeatable

1. **Rebuild:** Command Palette → **Dev Containers: Rebuild Container**, then repeat the tests in
   5.1. If everything works with no manual steps, your *setup files* are complete.
2. **Start from nothing (the real test):** open a *new, empty folder* and follow notebooks 01 and
   02 from the start, copying only the files. A new folder gets new volumes, so your working setup
   isn't affected. If this works, your *instructions* are complete too, and a colleague can
   follow them.

That's the whole tutorial: an agent, in an isolated container, connected to a real database and to
your own Python, with written rules, and set up so anyone can repeat it.

---
## Troubleshooting

These errors all happened while this tutorial was being written.

| Problem | Likely cause | What to do |
|---|---|---|
| `MP_API_KEY` is `MISSING` in the container, but `echo $MP_API_KEY` works on your computer | VS Code was started before the key was set, or from the Dock/Start menu. | Quit VS Code completely, start it from a terminal with `code .`, rebuild. |
| `error: pathspec 'v0.46.0' did not match` | The version name from the Materials Project docs doesn't exist in this copy of the code. | Use the commit hash from 2.2. |
| `pip install -e '.[mcp]'` says there is no `mcp` extra | You are on the main version, which has no MCP server. | Same fix: the commit hash from 2.2. |
| `cannot import name 'BSPathType'` | `emmet-core` 0.87 or newer was installed. | `~/mp-mcp/.venv/bin/pip install "emmet-core==0.86.4"` and keep the constraints file. |
| `'module://matplotlib_inline.backend_inline' is not a valid value for backend` in a check cell | Jupyter's plotting setting (`MPLBACKEND`) leaked into the server's Python. Only affects the notebook check, not the agent. | Use the updated check cell in 2.2, which removes that setting. |
| `postCreateCommand … failed with exit code 1` | The setup script stopped at an error, so later steps never ran. | Run `bash .devcontainer/setup-mp-mcp.sh` by hand to see the error. |
| `materials-project ✘ failed` in `/mcp` | Usually a wrong Python path in `.mcp.json`, or the server isn't installed. | Run the checks in 2.2 and 2.3; start the server by hand (2.4) to see the error. |
| Server connects, but every question gives an authentication error | The key is empty inside the container. | Run the check in 1.4. |
| Terminal seems frozen after starting the server by hand | The server is waiting for input; that's normal. | Press **Ctrl-D**. |
| Your own tool doesn't appear in `/mcp` | Missing `@mcp.tool` label, a JSON mistake in `.mcp.json`, or a wrong path. | Run the check cells in Stage 4; restart `claude`. |
| VS Code underlines `fastmcp` as "unable to import" | VS Code is looking at a different Python than the server uses. | Only cosmetic. To fix: *Python: Select Interpreter* → `~/mp-mcp/.venv/bin/python`. |
| Gmail / Drive / Calendar appear in `/mcp` without asking | Connectors from your claude.ai account. | Switch them off (3.3). |
| `Error: Connection closed` in the middle of a run | The server crashed or timed out. | Restart `claude`; `claude --debug` shows why. |

Nearly all the time spent on this setup went into ordinary software problems: a version name that
didn't exist, an unpinned package, a variable that didn't reach VS Code. None of it was about AI.
Warn the people you teach about this.

---
## Glossary

New words in this notebook (see notebook 01 for container, volume, rebuild, and so on):

| Term | Meaning |
|---|---|
| **API** | The entrance a service offers to *programs* instead of people. |
| **API key** | Your secret password for an API. Never share or commit it. |
| **Environment variable** | A named setting (e.g. `MP_API_KEY`) that programs can read. Good for secrets, because it stays out of files. |
| **Host** | Your own computer, as opposed to the container running on it. |
| **MCP** | *Model Context Protocol*: the standard "plug" for adding tools to agents. |
| **MCP server** | A small program that offers tools to agents through MCP. The agent starts it by itself. |
| **Client** | The side that uses the tools: the agent. |
| **`.mcp.json`** | The file that tells Claude Code which MCP servers to start and how. |
| **git clone / commit hash** | `git clone` downloads a code project. A commit hash (e.g. `553f725…`) names one exact version of that code. |
| **Virtual environment (venv)** | A separate Python installation with its own packages, so programs don't clash. |
| **Pinning / constraints file** | Fixing packages to exact versions so installs don't change over time. The constraints file is the list of those versions. |
| **Docstring** | The description in triple quotes at the start of a Python function. For a tool, it's what the agent reads. |
| **Decorator (`@mcp.tool`)** | A label above a function that changes how it's used. `@mcp.tool` means "offer this to the agent". |
| **`CLAUDE.md` / `GEMINI.md`** | Instructions the agent reads at the start of every session. A request, not a hard rule. |
| **`.claude/settings.json`** | Hard rules for Claude Code, such as blocked commands and files. Checked before every action. |
| **Connector** | A service (Gmail, Drive…) connected to your claude.ai account, which Claude Code can also use. |
| **POTCAR / `POTCAR.spec`** | VASP's licensed pseudopotential files / a list naming which ones to use, without the licensed data. |